# MLP FAO-56: aproximação da ETo calculada por Penman–Monteith

Este notebook compara redes neurais MLP para aproximar o alvo calculado `eto_pmt`, utilizando todas as combinações não vazias de Rn, u2, UR, Tmax, Tmin e Tmed: **63 cenários e uma única seed (42), totalizando 63 treinamentos**. O objetivo é aproximar a saída da fórmula FAO-56 Penman–Monteith na base estudada; o alvo não é uma medição independente de evapotranspiração em campo.

O protocolo utiliza divisão aleatória fixa em treino, validação e teste, normalização global das entradas e do alvo e seleção do cenário pelo menor RMSE de validação. O modelo final reutiliza os pesos da seed 42 do cenário escolhido, sem retreinamento. Os identificadores C01–C63, suas entradas e métricas são exportados em `tabela_mestre.csv`.

As tabelas, o modelo e a configuração são exportados em `resultados/dataframe/<execução>/`; as figuras, quando habilitadas, em `resultados/imagens/<execução>/`. Cada execução recebe um identificador exclusivo. O treinamento permanece em memória, sem checkpoints ou retomada automática.

**Tempo de referência informado para o treinamento dos 63 cenários: 202,9 minutos (aproximadamente 3 h 23 min).** Esse valor corresponde a uma execução anterior, não a uma medição desta cópia nem ao tempo das exportações. O hardware dessa medição não foi identificado neste registro; novas execuções registram seu próprio ambiente e duração.

Para reprodução, utilize o ambiente de `requirements.txt`, inicie o kernel na raiz do projeto e execute as células em ordem. Os gráficos estão preparados para os 63 cenários e permanecem opcionais por meio de `EXECUTAR_GRAFICOS = True`.


## 1. Ambiente computacional e importações


O ambiente de referência utiliza Python 3.12.14 e PyTorch 2.11.0+cu128 (CUDA 12.8). As dependências estão fixadas em `requirements.txt`; a instalação e a seleção do kernel do `.venv` estão descritas no README do repositório. O notebook não instala pacotes.

Execute a partir da raiz do repositório, onde os caminhos relativos dos dados e resultados são resolvidos. O dispositivo é selecionado automaticamente: CUDA local quando disponível, CPU caso contrário. O JSON registra o dispositivo e as versões efetivamente utilizados. Sementes fixas controlam a aleatoriedade do procedimento, mas não garantem resultados numericamente idênticos entre dispositivos ou versões de bibliotecas.


In [ ]:
import hashlib
import json
import platform
from importlib.metadata import version
from datetime import datetime, timezone

import copy  # usado para guardar uma cópia "congelada" dos melhores pesos do modelo

import openpyxl  # engine de escrita e leitura do Excel
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import re
import time
import unicodedata

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

# Semente inicial do PyTorch; cada treinamento redefine sua própria semente.
# A igualdade numérica entre ambientes e dispositivos não é garantida.
torch.manual_seed(42)

In [ ]:
# Caminhos relativos à raiz do repositório.
CAMINHO_DADOS = 'FAO-56/dados_FAO-56/dataset_A601_mlp.csv'
RAIZ_RESULTADOS = Path('resultados')
CAMINHO_MATRIZ_PEARSON = 'matriz_correlacao/resultados/matriz_correlacao_pearson.csv'
caminho_pearson = Path(CAMINHO_MATRIZ_PEARSON)
# Figuras dos 63 cenários; habilite para exportá-las após o treinamento.
EXECUTAR_GRAFICOS = True

# Excel é uma cópia de consulta dos CSVs; habilite apenas se necessário.
EXPORTAR_EXCEL = True


In [ ]:
# Usa a GPU NVIDIA deste computador quando CUDA está disponível; caso contrário, CPU.
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch: {torch.__version__}')
print(f'Dispositivo selecionado: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU local: {torch.cuda.get_device_name(DEVICE)}')
    print(f'CUDA do PyTorch: {torch.version.cuda}')
else:
    print('CUDA indisponível neste kernel. O processamento usará a CPU.')

# Pandas, gráficos e métricas usam CPU; os modelos e os lotes usam DEVICE.

## 2. Carregamento e validação dos dados

O arquivo `FAO-56/dados_FAO-56/dataset_A601_mlp.csv` é lido com separador vírgula. As sete colunas meteorológicas e do alvo são renomeadas em memória para os nomes usados no notebook; o CSV original é preservado. `rn_mj_m2_dia` corresponde a `Radiacao`, `u2_ms` a `Vel_Vento`, `ur_media_pct` a `umidade`, `tmax_c` a `Temp_Max`, `tmin_c` a `Temp_Min`, `tmedia_inmet_c` a `Temp_Med` e `eto_pmt` a `Eto_PMT`.

A coluna `data` é convertida para data e preservada em `datas_observacoes`, com os mesmos índices das linhas numéricas. Ela não entra na normalização nem nas entradas da MLP. A divisão continua aleatória, com seed 42. No CSV de previsões finais, a data identifica cada observação de teste.

Linhas com valores numéricos ausentes são removidas, e as datas são alinhadas às linhas restantes. Datas inválidas, colunas obrigatórias ausentes e valores não numéricos ou infinitos interrompem a execução. O hash SHA-256 identifica o CSV original; o total original de linhas é preservado em `n_linhas_original`.

Os nomes do novo CSV indicam saldo de radiação (Rn) em MJ/m²/dia, vento a 2 m em m/s, umidade em %, temperaturas em °C e ETo em mm/dia. O notebook apenas adapta o formato de entrada, sem converter unidades ou recalcular ETo.

### Proveniência e preparação

A base diária provém da estação automática INMET A601, Seropédica/Ecologia Agrícola. A versão documentada contém 8.329 dias completos, de 18/01/2001 a 30/08/2026, com lacunas. A preparação e o cálculo do alvo estão descritos em [FAO-56/README.md](FAO-56/README.md) e implementados em [penman–monteith.ipynb](FAO-56/penman–monteith.ipynb).

O processamento integra a radiação horária, calcula o saldo de radiação, converte vento de 10 m para 2 m e calcula a ETo diária com fluxo de calor no solo G = 0, sem imputação meteorológica. Tmed é a média fornecida pelo INMET. A documentação da base registra 31 dias com essa média inferior à temperatura mínima; esses registros são mantidos, sem exclusão adicional neste experimento.

Rn é uma variável derivada que já incorpora informação meteorológica. Remover uma temperatura das entradas da MLP, mantendo Rn, não elimina necessariamente toda a informação associada a essa temperatura. Consulte a documentação de preparação para os critérios e as limitações da base.


In [ ]:
# CSV A601: vírgula como separador e ponto como separador decimal.
df_original = pd.read_csv(CAMINHO_DADOS, sep=',')
with open(CAMINHO_DADOS, 'rb') as arquivo_dados:
    dataset_sha256 = hashlib.file_digest(arquivo_dados, 'sha256').hexdigest()
n_linhas_original = len(df_original)

COLUNAS_DATASET = {
    'rn_mj_m2_dia': 'Radiacao',
    'u2_ms': 'Vel_Vento',
    'ur_media_pct': 'umidade',
    'tmax_c': 'Temp_Max',
    'tmin_c': 'Temp_Min',
    'tmedia_inmet_c': 'Temp_Med',
    'eto_pmt': 'Eto_PMT',
}
faltantes = sorted(set(['data', *COLUNAS_DATASET]) - set(df_original.columns))
if faltantes:
    raise ValueError(f'Colunas obrigatórias ausentes no dataset A601: {faltantes}')

datas_observacoes = pd.to_datetime(df_original['data'], format='%Y-%m-%d', errors='raise')
if datas_observacoes.isna().any():
    raise ValueError('Datas ausentes no dataset A601.')
if datas_observacoes.duplicated().any():
    raise ValueError('Datas duplicadas no dataset diário A601.')
df = df_original[list(COLUNAS_DATASET)].rename(columns=COLUNAS_DATASET).copy()

print(f'Formato original: {df_original.shape[0]} linhas, {df_original.shape[1]} colunas')
print('Valores faltantes nas colunas do modelo:')
print(df.isna().sum())


In [ ]:
# Retirada de linhas com dados faltantes.
df = df.dropna(how='any')
datas_observacoes = datas_observacoes.loc[df.index].copy()

def validar_dados_numericos(tabela, nome):
    if not tabela.columns.is_unique:
        raise ValueError(f'{nome}: nomes de colunas duplicados.')
    if 'Eto_PMT' not in tabela.columns:
        raise ValueError(f'{nome}: coluna Eto_PMT ausente.')
    nao_numericas = [c for c in tabela if not pd.api.types.is_numeric_dtype(tabela[c])]
    if nao_numericas:
        raise ValueError(f'{nome}: colunas não numéricas: {nao_numericas}')
    if len(tabela) < 2:
        raise ValueError(f'{nome}: são necessárias pelo menos duas observações.')
    if not np.isfinite(tabela.to_numpy(dtype=float)).all():
        raise ValueError(f'{nome}: existem valores ausentes ou infinitos.')

validar_dados_numericos(df, 'Dados após limpeza')
if len(df) < 11:
    raise ValueError('São necessárias pelo menos 11 linhas para obter duas observações em cada partição.')

print(f'Formato após remover linhas com valores faltantes: {df.shape[0]} linhas, {df.shape[1]} colunas')

## 3. Divisão em treino, validação e teste

Duas chamadas a `train_test_split`, ambas com `random_state=42`, produzem aproximadamente 70% de treino, 15% de validação e 15% de teste. As mesmas observações compõem cada partição nos 63 cenários, todos treinados com a única seed 42.

A divisão aleatória é uma decisão metodológica alinhada ao objetivo de aproximar os valores calculados pela fórmula FAO-56 Penman–Monteith na distribuição da base estudada. O experimento avalia essa aproximação, sem estabelecer um protocolo de previsão de períodos futuros ou de validação contra evapotranspiração medida em campo.

A validação orienta o early stopping e a seleção do cenário. O teste é usado para avaliar os cenários e o modelo selecionado, sem escolher cenário, época ou hiperparâmetros. As três partições compartilham as estatísticas globais de normalização descritas a seguir.


In [ ]:
# 1a divisão: 70% treino, 30% "resto" (que ainda vira validação + teste)
df_train, df_temp = train_test_split(df, test_size=0.30, random_state=42)

# 2a divisão: divide o "resto" ao meio -> 15% validação, 15% teste
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=42)

for nome, conjunto in [('Treino', df_train), ('Validação', df_val), ('Teste', df_test)]:
    validar_dados_numericos(conjunto, nome)

print(f'Treino:    {len(df_train)} linhas ({len(df_train) / len(df):.1%})')
print(f'Validação: {len(df_val)} linhas ({len(df_val) / len(df):.1%})')
print(f'Teste:     {len(df_test)} linhas ({len(df_test) / len(df):.1%})')

## 4. Normalização com estatísticas do dataframe completo

Por decisão metodológica deste estudo, as entradas e o alvo são padronizados por z-score usando a média e o desvio-padrão amostral (`ddof=1`) de cada coluna do **dataframe completo após a remoção das linhas com valores ausentes** (`df`). As mesmas estatísticas globais são aplicadas aos conjuntos de treino, validação e teste.

Essa escolha inclui informações de validação e teste no ajuste da normalização, inclusive do alvo. Portanto, o pré-processamento não é independente desses conjuntos. O ajuste dos pesos continua usando apenas as amostras de treino, com early stopping na validação.

O MSE é calculado na escala normalizada. Para RMSE e R², as previsões e o alvo retornam à escala original usando as estatísticas globais de `Eto_PMT`. Desvios-padrão nulos ou não finitos impedem a execução.


In [ ]:
# Estatísticas por coluna do dataframe completo após a limpeza, conforme
# a decisão metodológica do estudo. São compartilhadas por treino, validação
# e teste e usadas também para desnormalizar as previsões (mm/dia).
mean = df.mean()
std = df.std(ddof=1)

invalidos = std.index[(~np.isfinite(std)) | (std <= 0)].tolist()
if invalidos:
    raise ValueError(f'Desvio-padrão do dataframe completo deve ser finito e positivo: {invalidos}')
if not np.isfinite(mean).all():
    raise ValueError('Médias do dataframe completo não finitas.')

df_train_norm = (df_train - mean) / std
df_val_norm = (df_val - mean) / std
df_test_norm = (df_test - mean) / std

for nome, conjunto in [('Treino normalizado', df_train_norm),
                       ('Validação normalizada', df_val_norm),
                       ('Teste normalizado', df_test_norm)]:
    validar_dados_numericos(conjunto, nome)
    if (np.abs(conjunto.to_numpy(dtype=float)) > np.finfo(np.float32).max).any():
        raise ValueError(f'{nome}: valores fora da faixa float32 usada pela rede.')


## 5. Experimentos: todas as combinações das seis variáveis

Uma camada oculta com 32 neurônios, ReLU e saída linear; Adam, learning rate 0,001, batch size 32, MSE, máximo de 10.000 épocas, patience 400 e restauração do primeiro mínimo de validação. Uma seed (42) por cenário. Apenas a dimensão da entrada varia, de uma a seis variáveis.

Nenhuma combinação não vazia é excluída. Pearson permanece como informação auxiliar e não seleciona os cenários.


In [ ]:
# Configuração científica atual: Uma repetição por cenário, seed 42.
N_REPETICOES = 1
SEMENTE_INICIAL = 42
SEMENTE_MODELO_FINAL = 42
TAMANHO_LOTE_EXPERIMENTO = 32
MAX_EPOCAS_EXPERIMENTO = 10000
PACIENCIA_EXPERIMENTO = 400
LEARNING_RATE_EXPERIMENTO = 0.001
HIDDEN_SIZE_EXPERIMENTO = 32

print(f'Repetições por cenário: {N_REPETICOES}')

### 5.1 Identificação das seis variáveis

Rn, u2, UR, Tmax, Tmin e Tmed participam das combinações; a filtragem permanece idêntica.


In [ ]:
def nome_simplificado(nome):
    texto = unicodedata.normalize('NFKD', str(nome))
    texto = ''.join(caractere for caractere in texto if not unicodedata.combining(caractere))
    return re.sub(r'[^a-z0-9]+', '', texto.lower())



# Exemplo com o nome interno: MAPEAMENTO_MANUAL['radiacao'] = 'Radiacao' (Rn)
MAPEAMENTO_MANUAL = {
    'radiacao': None,
    'vento': None,
    'umidade': None,
    'tmax': None,
    'tmin': None,
    'tmed': None,
}

ALIASES = {
    'radiacao': ['saldoderadiacao', 'radiacaoliquida', 'radiacao', 'rn'],
    'vento': ['velocidadevento', 'velvento', 'vento', 'wind', 'u2'],
    'umidade': ['umidaderelativa', 'umidade', 'humidity', 'rh', 'ur'],
    'tmax': ['temperaturamaxima', 'tempmaxima', 'tempmax', 'tmax'],
    'tmin': ['temperaturaminima', 'tempminima', 'tempmin', 'tmin'],
    'tmed': ['temperaturamedia', 'tempmedia', 'tempmed', 'tmedia', 'tmed'],
}


def resolver_coluna(chave, colunas, ja_usadas):
    manual = MAPEAMENTO_MANUAL[chave]
    if manual is not None:
        if manual not in colunas:
            raise ValueError(f'A coluna manual {manual!r} não existe no DataFrame.')
        if manual in ja_usadas:
            raise ValueError(f'A coluna {manual!r} foi associada a mais de uma variável.')
        return manual

    candidatas = []
    for coluna in colunas:
        if coluna in ja_usadas:
            continue
        simples = nome_simplificado(coluna)
        pontuacoes = []
        for alias in ALIASES[chave]:
            if simples == alias:
                pontuacoes.append(100 + len(alias))
            elif len(alias) >= 3 and alias in simples:
                pontuacoes.append(len(alias))
        if pontuacoes:
            candidatas.append((max(pontuacoes), coluna))

    if not candidatas:
        raise ValueError(
            f'Não foi possível identificar automaticamente {chave!r}. '
            f'Preencha MAPEAMENTO_MANUAL. Colunas disponíveis: {list(colunas)}'
        )

    candidatas.sort(reverse=True)
    melhor_pontuacao = candidatas[0][0]
    melhores = [coluna for pontuacao, coluna in candidatas if pontuacao == melhor_pontuacao]
    if len(melhores) > 1:
        raise ValueError(
            f'Mais de uma coluna pode representar {chave!r}: {melhores}. '
            'Escolha uma em MAPEAMENTO_MANUAL.'
        )
    return melhores[0]


colunas_entrada = df_train.drop(columns=['Eto_PMT']).columns.tolist()
mapeamento = {}
usadas = set()
for chave in ['radiacao', 'vento', 'umidade', 'tmax', 'tmin', 'tmed']:
    coluna = resolver_coluna(chave, colunas_entrada, usadas)
    mapeamento[chave] = coluna
    usadas.add(coluna)

if len(set(mapeamento.values())) != len(mapeamento):
    raise ValueError('Cada variável deve corresponder a uma coluna diferente.')

print('Mapeamento reconhecido:')
for chave, coluna in mapeamento.items():
    print(f'  {chave:8s} -> {coluna}')

ignoradas = [coluna for coluna in colunas_entrada if coluna not in usadas]
if ignoradas:
    print(f'\nAtenção: colunas não usadas nos cenários: {ignoradas}')


### 5.2 Definição determinística dos 63 cenários

São 6 combinações individuais, 15 pares, 20 trios, 15 quartetos, 6 quintetos e 1 conjunto completo. A enumeração C01–C63 segue primeiro o tamanho da combinação e depois a ordem Rn, u2, UR, Tmax, Tmin, Tmed.


In [ ]:
from itertools import combinations
from collections import Counter

chaves_candidatas = ['radiacao', 'vento', 'umidade', 'tmax', 'tmin', 'tmed']
variaveis_base = [mapeamento[chave] for chave in chaves_candidatas]
rotulos_variaveis = dict(zip(variaveis_base, ['Rn', 'u2', 'UR', 'Tmax', 'Tmin', 'Tmed']))
nomes_originais = {interno: original for original, interno in COLUNAS_DATASET.items()}
tipos_cenarios = {1: 'Individual', 2: 'Par', 3: 'Trio', 4: 'Quarteto', 5: 'Quinteto', 6: 'Completo'}

# Todas as combinações não vazias; ordem por quantidade e depois pela lista acima.
cenarios = {}
for n_entradas in range(1, 7):
    for combinacao in combinations(variaveis_base, n_entradas):
        nome = f'C{len(cenarios) + 1:02d}'
        cenarios[nome] = {
            'tipo': tipos_cenarios[n_entradas],
            'variaveis': list(combinacao),
            'rotulo': ' + '.join(rotulos_variaveis[v] for v in combinacao),
        }

assert len(cenarios) == 2**6 - 1 == 63
assert len({frozenset(c['variaveis']) for c in cenarios.values()}) == 63
assert Counter(len(c['variaveis']) for c in cenarios.values()) == {1: 6, 2: 15, 3: 20, 4: 15, 5: 6, 6: 1}
assert all('Eto_PMT' not in c['variaveis'] for c in cenarios.values())

df_cenarios = pd.DataFrame([
    {'cenario': nome, 'tipo_cenario': c['tipo'], 'rotulo': c['rotulo'],
     'n_variaveis': len(c['variaveis']),
     'variaveis': ' | '.join(nomes_originais[v] for v in c['variaveis'])}
    for nome, c in cenarios.items()
])
print(f'{len(cenarios)} cenários × {N_REPETICOES} seeds = {len(cenarios) * N_REPETICOES} treinamentos')
df_cenarios


In [ ]:
matriz_pearson = pd.read_csv(
    caminho_pearson,
    index_col=0,
    float_precision='round_trip',
)

# Aceita os nomes internos do notebook e os nomes originais do dataset.
matriz_pearson.rename(
    index=nomes_originais,
    columns=nomes_originais,
    inplace=True,
)

colunas_pearson = [
    nomes_originais[v] for v in variaveis_base
] + ['eto_pmt']

if (
    not matriz_pearson.index.is_unique
    or not matriz_pearson.columns.is_unique
):
    raise ValueError('Rótulos duplicados na matriz de Pearson.')

matriz_pearson = matriz_pearson.loc[
    colunas_pearson, colunas_pearson
].astype(float)

valores = matriz_pearson.to_numpy()

if not np.isfinite(valores).all():
    raise ValueError('A matriz contém valores ausentes ou infinitos.')

if (np.abs(valores) > 1.000001).any():
    raise ValueError('Existem coeficientes fora do intervalo [-1, 1].')

np.testing.assert_allclose(valores, valores.T, atol=1e-8)
np.testing.assert_allclose(np.diag(valores), 1, atol=1e-8)

# Confere a correspondência com a população efetivamente usada pela MLP.
pearson_dados = df.rename(columns=nomes_originais)[colunas_pearson].corr()
np.testing.assert_allclose(
    valores, pearson_dados.to_numpy(), rtol=1e-7, atol=1e-8,
    err_msg='A matriz de Pearson não corresponde ao dataset após limpeza.',
)
print(f'Matriz carregada: {CAMINHO_MATRIZ_PEARSON}')
matriz_pearson


### 5.3 Arquitetura da MLP

A rede possui uma camada oculta com 32 neurônios e ReLU, seguida de uma saída linear para regressão. O número de entradas é determinado pelo cenário. A instância exibida abaixo apresenta a arquitetura com duas entradas; os modelos experimentais são criados pela função de treinamento.


In [ ]:
class MLP(nn.Module):
    """MLP para regressão: entradas do cenário -> camada oculta ReLU -> saída linear.

    A camada de saída não tem função de ativação porque Eto_PMT é uma
    variável contínua (problema de regressão), não uma probabilidade/classe.
    """

    def __init__(self, input_size=2, hidden_size=32, output_size=1):
        super().__init__()
        self.camada_oculta1 = nn.Linear(input_size, hidden_size)
        self.camada_saida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.camada_oculta1(x))
        return self.camada_saida(x)


modelo = MLP().to(DEVICE)
modelo


### 5.4 Treinamento repetível e early stopping

Cada chamada inicializa a rede e o embaralhamento dos lotes com a semente informada. O treinamento minimiza MSE na escala normalizada, e as losses são ponderadas pelo número de observações de cada lote.

Uma melhora exige loss de validação **estritamente menor**. Os melhores pesos são copiados e restaurados ao término; a parada ocorre após 400 épocas consecutivas sem melhora ou no limite de 10.000 épocas. `melhor_epoca` registra o primeiro mínimo de validação e `epoca_parada` registra a última época executada.

A loss de treino agrega os lotes durante as atualizações dos pesos; a loss de validação é calculada ao final da época, sem gradientes. As curvas, portanto, não são avaliações simultâneas do mesmo estado da rede.


In [ ]:
def dataframe_para_tensores(df_normalizado, variaveis):
    x = torch.tensor(
        df_normalizado[variaveis].values.copy(),
        dtype=torch.float32,
    )
    y = torch.tensor(
        df_normalizado[['Eto_PMT']].values.copy(),
        dtype=torch.float32,
    )
    return x, y


def treinar_cenario(variaveis, semente):
    torch.manual_seed(semente)
    gerador_loader = torch.Generator().manual_seed(semente)

    x_treino, y_treino = dataframe_para_tensores(df_train_norm, variaveis)
    x_validacao, y_validacao = dataframe_para_tensores(df_val_norm, variaveis)
    x_teste, y_teste = dataframe_para_tensores(df_test_norm, variaveis)

    loader_treino_cenario = DataLoader(
        TensorDataset(x_treino, y_treino),
        batch_size=TAMANHO_LOTE_EXPERIMENTO,
        shuffle=True,
        generator=gerador_loader,
    )
    loader_validacao_cenario = DataLoader(
        TensorDataset(x_validacao, y_validacao),
        batch_size=TAMANHO_LOTE_EXPERIMENTO,
        shuffle=False,
    )

    modelo_cenario = MLP(
        input_size=len(variaveis),
        hidden_size=HIDDEN_SIZE_EXPERIMENTO,
    ).to(DEVICE)
    criterio = nn.MSELoss()
    otimizador = optim.Adam(
        modelo_cenario.parameters(),
        lr=LEARNING_RATE_EXPERIMENTO,
    )

    melhor_loss_validacao = float('inf')
    melhor_epoca = 0
    melhores_pesos_cenario = None
    epocas_sem_melhoria = 0
    historico_treino_cenario = []
    historico_validacao_cenario = []

    for epoca in range(1, MAX_EPOCAS_EXPERIMENTO + 1):
        modelo_cenario.train()
        soma_loss_treino = 0.0
        n_treino = 0
        for x_lote, y_lote in loader_treino_cenario:
            x_lote = x_lote.to(DEVICE)
            y_lote = y_lote.to(DEVICE)
            otimizador.zero_grad()
            previsoes = modelo_cenario(x_lote)
            loss = criterio(previsoes, y_lote)
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f'Loss de treino não finita: seed={semente}, época={epoca}, entradas={variaveis}')
            loss.backward()
            otimizador.step()
            soma_loss_treino += loss.item() * len(x_lote)
            n_treino += len(x_lote)

        loss_treino_epoca = soma_loss_treino / n_treino

        modelo_cenario.eval()
        soma_loss_validacao = 0.0
        n_validacao = 0
        with torch.no_grad():
            for x_lote, y_lote in loader_validacao_cenario:
                x_lote = x_lote.to(DEVICE)
                y_lote = y_lote.to(DEVICE)
                loss_lote = criterio(modelo_cenario(x_lote), y_lote)
                soma_loss_validacao += loss_lote.item() * len(x_lote)
                n_validacao += len(x_lote)

        loss_validacao_epoca = soma_loss_validacao / n_validacao
        if not np.isfinite([loss_treino_epoca, loss_validacao_epoca]).all():
            raise FloatingPointError(f'Loss não finita: seed={semente}, época={epoca}, entradas={variaveis}')
        historico_treino_cenario.append(loss_treino_epoca)
        historico_validacao_cenario.append(loss_validacao_epoca)

        if loss_validacao_epoca < melhor_loss_validacao:
            melhor_loss_validacao = loss_validacao_epoca
            melhor_epoca = epoca
            melhores_pesos_cenario = copy.deepcopy(modelo_cenario.state_dict())
            epocas_sem_melhoria = 0
        else:
            epocas_sem_melhoria += 1

        if epocas_sem_melhoria >= PACIENCIA_EXPERIMENTO:
            break

    if melhores_pesos_cenario is None:
        raise RuntimeError('Treinamento sem pesos válidos para restauração.')
    modelo_cenario.load_state_dict(melhores_pesos_cenario)
    modelo_cenario.eval()

    with torch.no_grad():
        previsao_validacao_norm = (
            modelo_cenario(x_validacao.to(DEVICE))
            .cpu()
            .numpy()
            .ravel()
        )

        previsao_teste_norm = (
             modelo_cenario(x_teste.to(DEVICE))
            .cpu()
            .numpy()
            .ravel()
        )


    real_validacao_norm = y_validacao.numpy().ravel()
    real_teste_norm = y_teste.numpy().ravel()
    eto_media = mean['Eto_PMT']
    eto_desvio = std['Eto_PMT']
    previsao_validacao = previsao_validacao_norm * eto_desvio + eto_media
    real_validacao = real_validacao_norm * eto_desvio + eto_media
    previsao_teste = previsao_teste_norm * eto_desvio + eto_media
    real_teste = real_teste_norm * eto_desvio + eto_media

    metricas = {
        # Validação
        'mse_valid_norm': melhor_loss_validacao,
        'rmse_valid_mm_dia': root_mean_squared_error(
          real_validacao,
          previsao_validacao,
        ),
        'r2_valid': r2_score(
           real_validacao,
          previsao_validacao,
        ),

        # Teste
        'mse_teste_norm': float(
         np.mean((previsao_teste_norm - real_teste_norm) ** 2)
        ),
        'rmse_teste_mm_dia': root_mean_squared_error(
          real_teste,
         previsao_teste,
        ),
        'r2_teste': r2_score(
          real_teste,
          previsao_teste,
        ),

     # Treinamento
        'melhor_epoca': melhor_epoca,
        'epoca_parada': epoca,
        'epocas': epoca,
    }
    historico = {
        'treino': historico_treino_cenario,
        'validacao': historico_validacao_cenario,
    }
    return modelo_cenario, metricas, historico


### 5.5 Execução dos 63 treinamentos

O laço chama diretamente `treinar_cenario(variaveis, semente)` para cada cenário e seed. Métricas, históricos e pesos da seed 42 ficam em memória, sem gravar arquivos durante os treinamentos. Todos os cenários compartilham o split e a normalização já calculados. As exportações acontecem nas seções seguintes, após a conclusão dos 63 treinamentos.

Executar novamente esta célula refaz todos os treinamentos. Não há leitura de checkpoints nem retomada automática.


In [ ]:
resultados = []
historicos = []
pesos_seed_modelo_final = {}
sementes = list(range(SEMENTE_INICIAL, SEMENTE_INICIAL + N_REPETICOES))
total_treinamentos = len(cenarios) * len(sementes)
if sementes != [42] or SEMENTE_MODELO_FINAL != 42:
    raise ValueError('O protocolo deste estudo requer uma única seed: 42.')
if min(TAMANHO_LOTE_EXPERIMENTO, MAX_EPOCAS_EXPERIMENTO, PACIENCIA_EXPERIMENTO, HIDDEN_SIZE_EXPERIMENTO) < 1:
    raise ValueError('Batch size, épocas, paciência e dimensão oculta devem ser positivos.')
if not np.isfinite(LEARNING_RATE_EXPERIMENTO) or LEARNING_RATE_EXPERIMENTO <= 0:
    raise ValueError('Learning rate deve ser finito e positivo.')

# Configuração e identificação dos arquivos capturadas antes dos treinamentos.
caminho_notebook = Path('mlp_FAO-56.ipynb')
notebook_sha256 = hashlib.sha256(caminho_notebook.read_bytes()).hexdigest()
configuracao_execucao = {
    'notebook': {'arquivo': caminho_notebook.name, 'sha256': notebook_sha256},
    'tempo_referencia_informado_minutos': 202.9,
    'inicio_utc': datetime.now(timezone.utc).isoformat(),
    'hiperparametros': {
        'n_repeticoes': N_REPETICOES, 'semente_inicial': SEMENTE_INICIAL,
        'sementes': sementes.copy(), 'batch_size': TAMANHO_LOTE_EXPERIMENTO,
        'hidden_size': HIDDEN_SIZE_EXPERIMENTO, 'learning_rate': LEARNING_RATE_EXPERIMENTO,
        'max_epocas': MAX_EPOCAS_EXPERIMENTO, 'paciencia': PACIENCIA_EXPERIMENTO,
    },
    'modelo': {'camadas_ocultas': 1, 'ativacao': 'ReLU', 'otimizador': 'Adam', 'loss': 'MSELoss'},
    'dispositivo': str(DEVICE),
    'gpu': torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else None,
    'cuda_pytorch': torch.version.cuda,
    'versoes': {'python': platform.python_version(), **{
        pacote: version(pacote) for pacote in
        ['torch', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'openpyxl']
    }},
    'dataset': {'sha256': dataset_sha256, 'linhas_originais': n_linhas_original,
                'linhas_apos_limpeza': len(df), 'colunas': df.columns.tolist()},
    'divisao': {'metodo': 'train_test_split', 'random_state': 42,
                'test_size_primeira_divisao': 0.30, 'test_size_segunda_divisao': 0.50,
                'n_treino': len(df_train), 'n_validacao': len(df_val), 'n_teste': len(df_test)},
    'normalizacao': 'z-score com média e desvio amostral do dataframe completo após limpeza (ddof=1)',
    'quantidade_cenarios': len(cenarios),
    'cenarios': copy.deepcopy(cenarios),
}

contador = 0
inicio = time.time()

for nome_cenario, config in cenarios.items():
    
    for semente in sementes:
        contador += 1
        inicio_cenario = time.time()
        modelo_treinado, metricas, historico = treinar_cenario(config['variaveis'], semente)
        tempo_treinamento_segundos = time.time() - inicio_cenario
        resultados.append({
            'cenario': nome_cenario,
            'tipo': config['tipo'],
            'n_variaveis': len(config['variaveis']),
            'variaveis': ' | '.join(config['variaveis']),
            'semente': semente,
            **metricas,
            'tempo_treinamento_segundos': tempo_treinamento_segundos,
        })

        if semente == SEMENTE_MODELO_FINAL:
            pesos_seed_modelo_final[nome_cenario] = copy.deepcopy(
             modelo_treinado.state_dict()
                                )

        for epoca, (loss_treino, loss_validacao) in enumerate(
            zip(historico['treino'], historico['validacao'], strict=True), start=1,
        ):
            historicos.append({
                'cenario': nome_cenario,
                'tipo': config['tipo'],
                'semente': semente,
                'epoca': epoca,
                'loss_treino': loss_treino,
                'loss_validacao': loss_validacao,
            })
        decorrido_min = (time.time() - inicio) / 60
        print(
            f'[{contador:02d}/{total_treinamentos}] {' | '.join(config['variaveis'])} | '
            f'semente={semente} | RMSE val={metricas["rmse_valid_mm_dia"]:.4f} '
            f'mm/dia | R²={metricas["r2_valid"]:.4f} | '
            f'tempo={tempo_treinamento_segundos:.1f}s | {decorrido_min:.1f} min'
        )

df_resultados = pd.DataFrame(resultados)
df_loss_epocas = pd.DataFrame(historicos)
df_tempos_treinamento = df_resultados[
    ['cenario', 'tipo', 'variaveis', 'semente', 'tempo_treinamento_segundos']
].copy()
df_tempos_treinamento['tempo_treinamento_minutos'] = (
    df_tempos_treinamento['tempo_treinamento_segundos'] / 60
)


configuracao_execucao['duracao_treinamentos_minutos'] = (time.time() - inicio) / 60
configuracao_execucao['fim_treinamentos_utc'] = datetime.now(timezone.utc).isoformat()
print(f"Concluído em {configuracao_execucao['duracao_treinamentos_minutos']:.1f} minutos.")
df_resultados.head()



### 5.6 Integridade dos históricos

Os registros são verificados quanto aos 63 cenários com seed 42, sequência de épocas, época de parada, melhor época e correspondência entre mínimo de validação e MSE registrado.

Cada curva representa o único treinamento do respectivo cenário e termina na época efetivamente executada. O histórico é preservado uma única vez, em `loss_epocas.csv`, sem tabelas de médias ou desvios entre seeds.


In [ ]:
def validar_registros_experimento(repeticoes, losses, cenarios_esperados, sementes_esperadas):
    chaves = ['cenario', 'semente']
    esperados = {(cenario, semente) for cenario in cenarios_esperados for semente in sementes_esperadas}
    assert len(repeticoes) == len(cenarios_esperados) * len(sementes_esperadas), 'Quantidade de treinamentos incorreta.'
    assert not repeticoes.duplicated(chaves).any(), 'Treinamentos duplicados.'
    assert set(repeticoes[chaves].itertuples(index=False, name=None)) == esperados, 'Cenários/sementes divergentes.'
    assert not losses.duplicated(chaves + ['epoca']).any(), 'Épocas duplicadas.'
    assert set(losses[chaves].itertuples(index=False, name=None)) == esperados, 'Históricos ausentes ou extras.'
    assert np.isfinite(losses[['loss_treino', 'loss_validacao']].to_numpy()).all(), 'Loss não finita.'

    grupos = losses.groupby(chaves, sort=False)
    for repeticao in repeticoes.itertuples(index=False):
        curva = grupos.get_group((repeticao.cenario, repeticao.semente)).sort_values('epoca')
        assert len(curva) == repeticao.epoca_parada, 'Histórico incompleto.'
        assert curva['epoca'].tolist() == list(range(1, int(repeticao.epoca_parada) + 1)), 'Sequência de épocas inválida.'
        assert 1 <= repeticao.melhor_epoca <= repeticao.epoca_parada, 'Melhor época inválida.'
        assert curva['tipo'].eq(repeticao.tipo).all(), 'Tipo de cenário divergente.'
        assert np.isclose(curva['loss_validacao'].min(), repeticao.mse_valid_norm, rtol=1e-7, atol=1e-10), 'MSE mínimo divergente.'
        primeira_melhor = curva.loc[curva['loss_validacao'].idxmin(), 'epoca']
        assert primeira_melhor == repeticao.melhor_epoca, 'Melhor época não corresponde ao primeiro mínimo.'


validar_registros_experimento(df_resultados, df_loss_epocas, cenarios, sementes)

print(f'Registros verificados: {len(df_resultados)} treinamentos e {len(df_loss_epocas)} épocas.')


def tabela_mestre(registros):
    tabela = pd.DataFrame(registros).rename(columns={
        'tipo': 'tipo_cenario', 'semente': 'seed',
        'rmse_valid_mm_dia': 'rmse_validacao', 'r2_valid': 'r2_validacao',
        'rmse_teste_mm_dia': 'rmse_teste',
        'mse_valid_norm': 'mse_validacao_normalizado',
        'mse_teste_norm': 'mse_teste_normalizado',
    })
    tabela['variaveis'] = tabela['cenario'].map(df_cenarios.set_index('cenario')['variaveis'])
    # Uma linha por cenário: validação e teste lado a lado.
    tabela = tabela.drop(columns=['epocas'])  # alias de epoca_parada
    tabela['rotulo'] = tabela['cenario'].map(df_cenarios.set_index('cenario')['rotulo'])
    return tabela.sort_values(['rmse_validacao', 'cenario'], kind='stable').reset_index(drop=True)

df_mestre = tabela_mestre(resultados)


## 6. Resumo dos 63 cenários — seed 42

`tabela_mestre.csv` reúne entradas, seed, métricas de validação e teste e épocas de cada cenário. A ordenação usa exclusivamente RMSE de validação, com desempate pela ordem C01–C63. Como há uma única seed, não se exportam médias e desvios entre repetições.


In [ ]:
# Exportação após os 63 treinamentos, sem salvamento durante o treinamento.
from uuid import uuid4
ID_EXECUCAO = datetime.now(timezone.utc).strftime('execucao_%Y%m%dT%H%M%S_%fZ_') + uuid4().hex[:8]
PASTA_RESULTADOS = RAIZ_RESULTADOS
PASTA_DATAFRAME = PASTA_RESULTADOS / 'dataframe' / ID_EXECUCAO
PASTA_IMAGENS = PASTA_RESULTADOS / 'imagens' / ID_EXECUCAO
PASTA_DATAFRAME.mkdir(parents=True, exist_ok=False)
PASTA_IMAGENS.mkdir(parents=True, exist_ok=False)
print(f'Tabelas: {PASTA_DATAFRAME}')
print(f'Imagens: {PASTA_IMAGENS}')

df_mestre.to_csv(PASTA_DATAFRAME / 'tabela_mestre.csv', index=False)
df_loss_epocas.to_csv(PASTA_DATAFRAME / 'loss_epocas.csv', index=False)
df_normalizacao = pd.DataFrame({
    'variavel': mean.index, 'media': mean.to_numpy(), 'desvio_amostral': std.to_numpy(),
})
df_normalizacao.to_csv(PASTA_DATAFRAME / 'normalizacao.csv', index=False)
df_split = pd.concat([
    pd.DataFrame({'indice_original': conjunto.index,
                  'data': datas_observacoes.loc[conjunto.index].dt.strftime('%Y-%m-%d').to_numpy(),
                  'particao': nome, 'ordem': np.arange(len(conjunto))})
    for nome, conjunto in [('treino', df_train), ('validacao', df_val), ('teste', df_test)]
], ignore_index=True)
df_split.to_csv(PASTA_DATAFRAME / 'split.csv', index=False)

# Visões em memória para os gráficos e a seleção; não geram arquivos adicionais.
resumo = df_mestre.rename(columns={
    'tipo_cenario': 'tipo', 'rmse_validacao': 'rmse_medio', 'r2_validacao': 'r2_medio',
})
resumo_teste = df_mestre.rename(columns={
    'tipo_cenario': 'tipo', 'rmse_teste': 'rmse_medio', 'r2_teste': 'r2_medio',
})

# Associações específicas dos pares; as métricas estão na tabela mestre.
linhas_pares = []
for nome, config in cenarios.items():
    if config['tipo'] != 'Par':
        continue
    v1, v2 = [nomes_originais[v] for v in config['variaveis']]
    linhas_pares.append({
        'cenario': nome, 'variavel_1': v1, 'variavel_2': v2,
        'abs_r_variavel_1_eto': abs(matriz_pearson.loc[v1, 'eto_pmt']),
        'abs_r_variavel_2_eto': abs(matriz_pearson.loc[v2, 'eto_pmt']),
        'r_entre_variaveis': matriz_pearson.loc[v1, v2],
    })
tabela_pearson_pares = pd.DataFrame(linhas_pares)
tabela_pearson_pares.to_csv(PASTA_DATAFRAME / 'pearson_pares.csv', index=False)
df_tempos_treinamento.to_csv(PASTA_DATAFRAME / 'tempos_treinamento.csv', index=False)


## 7. Gráficos comparativos

Figuras opcionais de RMSE de teste, divididas por tamanho do conjunto de entradas (Individual, Par, Trio, Quarteto e Quinteto). Cada gráfico compara os cenários daquele tamanho com o modelo completo (6 variáveis), incluído como barra de referência em todos eles.


In [ ]:
if EXECUTAR_GRAFICOS:
    cores_tipo = {
        'Individual': '#0072B2', 'Par': '#009E73', 'Trio': '#E69F00',
        'Quarteto': '#CC79A7', 'Quinteto': '#D55E00', 'Completo': '#555555',
    }
    for tabela, metrica, titulo, arquivo in [
        (resumo, 'rmse_medio', 'RMSE de validação (mm/dia)', 'rmse_validacao'),
        (resumo_teste, 'rmse_medio', 'RMSE de teste (mm/dia)', 'rmse_teste'),
        (resumo_teste, 'r2_medio', 'R² de teste', 'r2_teste'),
    ]:
        fig, eixo = plt.subplots(figsize=(12, 20), constrained_layout=True)
        posicoes = np.arange(len(tabela))
        eixo.barh(posicoes, tabela[metrica],
                  color=[cores_tipo[t] for t in tabela['tipo']])
        eixo.set_yticks(posicoes)
        eixo.set_yticklabels([f"{c}: {cenarios[c]['rotulo']}" for c in tabela['cenario']], fontsize=9)
        eixo.invert_yaxis()
        eixo.set_xlabel(titulo)
        eixo.grid(axis='x', alpha=0.25)
        from matplotlib.patches import Patch
        eixo.legend(handles=[Patch(color=cor, label=tipo) for tipo, cor in cores_tipo.items()],
                    loc='lower right', title='Tipo de cenário')
        for extensao in ['png']:
            fig.savefig(PASTA_IMAGENS / f'{arquivo}.{extensao}', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)
else:
    print('Gráficos opcionais desativados: EXECUTAR_GRAFICOS = False.')


In [ ]:
if EXECUTAR_GRAFICOS:
    cores_tipo = {
        'Individual': '#0072B2', 'Par': '#009E73', 'Trio': '#E69F00',
        'Quarteto': '#CC79A7', 'Quinteto': '#D55E00', 'Completo': '#555555',
    }
    from matplotlib.patches import Patch

    # Linha do modelo completo (6 variáveis), repetida como referência em cada gráfico.
    linha_completo = resumo_teste.loc[resumo_teste['tipo'] == 'Completo'].iloc[[0]]

    for tipo in ['Individual', 'Par', 'Trio', 'Quarteto', 'Quinteto']:
        tabela_tipo = resumo_teste.loc[resumo_teste['tipo'] == tipo].sort_values(
            ['rmse_medio', 'cenario'], kind='stable',
        )
        tabela = pd.concat([tabela_tipo, linha_completo], ignore_index=True)

        fig, eixo = plt.subplots(
            figsize=(12, 0.4 * len(tabela) + 1.5), constrained_layout=True,
        )
        posicoes = np.arange(len(tabela))
        eixo.barh(posicoes, tabela['rmse_medio'],
                  color=[cores_tipo[t] for t in tabela['tipo']])
        eixo.bar_label(eixo.containers[-1], fmt='%.4f')
        eixo.set_yticks(posicoes)
        eixo.set_yticklabels([f"{c}: {cenarios[c]['rotulo']}" for c in tabela['cenario']], fontsize=9)
        eixo.invert_yaxis()
        eixo.set_xlabel('RMSE de teste (mm/dia)')
        eixo.grid(axis='x', alpha=0.25)
     

        arquivo = f'rmse_teste_{tipo.lower()}'
        for extensao in ['png']:
            fig.savefig(PASTA_IMAGENS / f'{arquivo}.{extensao}', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)
else:
    print('Gráficos opcionais desativados: EXECUTAR_GRAFICOS = False.')


In [ ]:
# A comparação exaustiva dos cenários é apresentada nas figuras da seção 7.


## 8. Seleção pela validação e avaliação no teste

Seleciona-se o menor RMSE médio de validação entre os 63 cenários; empates exatos seguem a ordem C01–C63. O modelo representativo é sempre a seed 42 já treinada do cenário escolhido, sem seleção da melhor seed e sem novo treinamento. O teste não escolhe cenário, época nem hiperparâmetros.


In [ ]:
CENARIO_FINAL = resumo.iloc[0]['cenario']
selecao_automatica = True

variaveis_finais = cenarios[CENARIO_FINAL]['variaveis']
print(f'Cenário final: {CENARIO_FINAL}')
print(f'Variáveis: {variaveis_finais}')

modelo_final = MLP(
    input_size=len(variaveis_finais),
    hidden_size=HIDDEN_SIZE_EXPERIMENTO,
).to(DEVICE)

modelo_final.load_state_dict(
    pesos_seed_modelo_final[CENARIO_FINAL]
)

modelo_final.eval()
linha_modelo_final = df_resultados.loc[
    df_resultados['cenario'].eq(CENARIO_FINAL)
    & df_resultados['semente'].eq(SEMENTE_MODELO_FINAL)
].iloc[0]

metricas_validacao_final = {
    'mse_valid_norm': linha_modelo_final['mse_valid_norm'],
    'rmse_valid_mm_dia': linha_modelo_final['rmse_valid_mm_dia'],
    'r2_valid': linha_modelo_final['r2_valid'],
    'melhor_epoca': int(linha_modelo_final['melhor_epoca']),
    'epoca_parada': int(linha_modelo_final['epoca_parada']),
}
print(f'Métricas de validação: {metricas_validacao_final}')
configuracao_execucao['modelo_final'] = {
    'cenario': CENARIO_FINAL,
    'semente': SEMENTE_MODELO_FINAL,
    'selecao': (
        'menor_rmse_medio_validacao'
        if selecao_automatica
        else 'manual'
    ),
}


In [ ]:
x_teste_final, y_teste_final = dataframe_para_tensores(
    df_test_norm,
    variaveis_finais,
)

modelo_final.eval()
with torch.no_grad():
    previsao_teste_norm = (
        modelo_final(x_teste_final.to(DEVICE))
        .cpu()
        .numpy()
        .ravel()
    )

real_teste_norm = y_teste_final.numpy().ravel()
eto_media = mean['Eto_PMT']
eto_desvio = std['Eto_PMT']
previsao_teste = previsao_teste_norm * eto_desvio + eto_media
real_teste = real_teste_norm * eto_desvio + eto_media

metricas_teste_final = {
    'cenario': CENARIO_FINAL,
    'semente': SEMENTE_MODELO_FINAL,
    'n_variaveis': len(variaveis_finais),
    'mse_teste_norm': float(np.mean((previsao_teste_norm - real_teste_norm) ** 2)),
    'rmse_teste_mm_dia': root_mean_squared_error(real_teste, previsao_teste),
    'r2_teste': r2_score(real_teste, previsao_teste),
}

print('Avaliação final no teste:')
for nome, valor in metricas_teste_final.items():
    if isinstance(valor, float):
        print(f'  {nome}: {valor:.6f}')
    else:
        print(f'  {nome}: {valor}')


## 9. Dispersão entre ETo calculada e ETo aproximada

Figura opcional do modelo completo, melhor modelo reduzido e pior modelo reduzido.  


In [ ]:
if EXECUTAR_GRAFICOS:
    import textwrap

    # Identifica o único cenário com as seis entradas.
    completos = [
        nome for nome, config in cenarios.items()
        if len(config['variaveis']) == 6
    ]
    if len(completos) != 1:
        raise ValueError('Deve existir exatamente um cenário completo.')

    nome_completo = completos[0]

    # "resumo" contém o RMSE médio de VALIDAÇÃO.
    # Com apenas a seed 42, essa média é o valor da própria seed.
    reduzidos = resumo.loc[
        resumo['cenario'].isin(cenarios)
        & resumo['cenario'].ne(nome_completo)
    ].copy()

    if len(reduzidos) < 2:
        raise ValueError('São necessários pelo menos dois cenários reduzidos.')

    if not np.isfinite(reduzidos['rmse_medio'].to_numpy()).all():
        raise ValueError('Há RMSEs de validação ausentes ou inválidos.')

    reduzidos['ordem'] = reduzidos['cenario'].map(
        {nome: i for i, nome in enumerate(cenarios)}
    )

    nome_melhor = reduzidos.sort_values(
        ['rmse_medio', 'ordem'],
        ascending=[True, True],
    ).iloc[0]['cenario']

    nome_pior = reduzidos.sort_values(
        ['rmse_medio', 'ordem'],
        ascending=[False, True],
    ).iloc[0]['cenario']

    selecionados = [
        ('Completo', nome_completo),
        ('Melhor reduzido', nome_melhor),
        ('Pior reduzido', nome_pior),
    ]

    # Calcula previsões somente para os cenários selecionados.
    previsoes_cenarios = {}
    referencia = None

    for nome in dict.fromkeys(nome for _, nome in selecionados):
        config = cenarios[nome]

        modelo_cenario = MLP(
            input_size=len(config['variaveis']),
            hidden_size=HIDDEN_SIZE_EXPERIMENTO,
        ).to(DEVICE)

        modelo_cenario.load_state_dict(pesos_seed_modelo_final[nome])
        modelo_cenario.eval()

        x_cenario, y_cenario = dataframe_para_tensores(
            df_test_norm,
            config['variaveis'],
        )

        with torch.no_grad():
            previsto_norm = (
                modelo_cenario(x_cenario.to(DEVICE))
                .cpu()
                .numpy()
                .ravel()
            )

        previsoes_cenarios[nome] = (
            previsto_norm * std['Eto_PMT'] + mean['Eto_PMT']
        )

        referencia_cenario = (
            y_cenario.numpy().ravel()
            * std['Eto_PMT'] + mean['Eto_PMT']
        )

        if referencia is None:
            referencia = referencia_cenario
        else:
            np.testing.assert_array_equal(
                referencia, referencia_cenario
            )

    # Mesma escala nos três painéis.
    valores = np.concatenate([
        referencia, *previsoes_cenarios.values()
    ])
    margem = max(float(np.ptp(valores)) * 0.05, 0.1)
    limites = (
        float(valores.min()) - margem,
        float(valores.max()) + margem,
    )

    fig, eixos = plt.subplots(
        1, 3,
        figsize=(16, 6),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for eixo, (titulo, nome) in zip(eixos, selecionados):
        previsto = previsoes_cenarios[nome]
        rmse = root_mean_squared_error(referencia, previsto)
        r2 = r2_score(referencia, previsto)

        eixo.scatter(
            referencia, previsto,
            alpha=0.4,
            s=14,
            color='#2878A5',
            edgecolors='none',
            rasterized=True,
        )
        eixo.plot(
            limites, limites,
            '--',
            color='firebrick',
            linewidth=1.2,
        )

        eixo.set_title(cenarios[nome]['rotulo'],
        fontsize=11,loc='left',)
        eixo.set(xlim=limites, ylim=limites)
        eixo.set_aspect('equal', adjustable='box')
        eixo.tick_params(labelsize=11)
        eixo.grid(alpha=0.15)

        eixo.text(
            0.04, 0.96,
            f'R² = {r2:.4f}\nRMSE = {rmse:.4f} mm/dia',
            transform=eixo.transAxes,
            va='top',
            fontsize=11,
            bbox=dict(
                facecolor='white',
                edgecolor='none',
                alpha=0.85,
            ),
        )

    fig.supxlabel('ETo Penman–Monteith (mm/dia)', fontsize=13)
    fig.supylabel('ETo MLP (mm/dia)', fontsize=13)

    PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)

    for extensao in ['png']:
        fig.savefig(
            PASTA_IMAGENS
            / f'figura_dispersao_cenarios_representativos.{extensao}',
            dpi=300,
            bbox_inches='tight',
        )

    plt.show()
    plt.close(fig)

else:
    print('Gráficos opcionais desativados: EXECUTAR_GRAFICOS = False.')

In [ ]:
residuos_teste = previsao_teste - real_teste

df_previsoes_teste = pd.DataFrame({
    'data': datas_observacoes.loc[df_test.index].dt.strftime('%Y-%m-%d').to_numpy(),
    'eto_real_mm_dia': real_teste,
    'eto_prevista_mm_dia': previsao_teste,
    'residuo_mm_dia': residuos_teste,
})
df_previsoes_teste.to_csv(
    PASTA_DATAFRAME / 'previsoes_modelo_final_teste.csv',
    index=False,
)

torch.save(
    {
        'state_dict': modelo_final.state_dict(),
        'cenario': CENARIO_FINAL,
        'variaveis': variaveis_finais,
        'hidden_size': HIDDEN_SIZE_EXPERIMENTO,
        'media_normalizacao': mean.to_dict(),
        'desvio_normalizacao': std.to_dict(),
        'origem_normalizacao': 'dataframe_completo_apos_limpeza',
        'ddof_normalizacao': 1,
        'metricas_validacao': metricas_validacao_final,
        'metricas_teste': metricas_teste_final,
    },
    PASTA_DATAFRAME / 'modelo_final_evapotranspiracao.pt',
)

print('Arquivos presentes na pasta nesta etapa:')
for arquivo in sorted(PASTA_DATAFRAME.iterdir()):
    print(f'  {arquivo.name}')

configuracao_execucao['status'] = 'exportacoes_em_andamento'
configuracao_execucao['metricas_modelo_final_validacao'] = metricas_validacao_final
configuracao_execucao['metricas_modelo_final_teste'] = metricas_teste_final
(PASTA_DATAFRAME / 'configuracao_execucao.json').write_text(
    json.dumps(configuracao_execucao, ensure_ascii=False, indent=2, allow_nan=False),
    encoding='utf-8',
)


### 9.1 Exportação e verificação

Cada execução produz seis CSVs em `resultados_completos/dataframe/<execução>/`:

- `tabela_mestre.csv`: 63 cenários, entradas, seed, métricas de validação e teste e épocas.
- `loss_epocas.csv`: histórico completo por cenário e época.
- `pearson_pares.csv`: correlações dos 15 pares; relacione por `cenario` com a tabela mestre.
- `previsoes_modelo_final_teste.csv`: datas, referências, previsões e resíduos do modelo escolhido.
- `normalizacao.csv`: médias e desvios utilizados no pré-processamento.
- `split.csv`: associação das observações às partições, com sua ordem original.
- `tempos_treinamento.csv`: tempo de treinamento de cada combinação.

O modelo `.pt` conserva pesos e parâmetros necessários para inferência. `configuracao_execucao.json` registra ambiente, protocolo, cenário escolhido e suas métricas. Essa informação também acompanha o modelo para permitir seu uso isolado.

`EXPORTAR_EXCEL = True` gera, adicionalmente, um único `resultados.xlsx`, com abas de cenários, Pearson e previsões. É uma cópia de consulta opcional; o histórico extenso permanece apenas no CSV. Por padrão, nenhum Excel é gerado. Os CSVs e, quando habilitado, o Excel são relidos e conferidos.

A pasta `resultados_completos/imagens/<execução>/` recebe as figuras quando os gráficos forem ativados. Exportações de execuções anteriores não são apagadas.


In [ ]:
# Confere todas as tabelas exportadas antes de marcar a execução como concluída.
validar_registros_experimento(df_resultados, df_loss_epocas, cenarios, sementes)
csv_esperados = {
    'tabela_mestre.csv': df_mestre,
    'loss_epocas.csv': df_loss_epocas,
    'normalizacao.csv': df_normalizacao,
    'split.csv': df_split,
    'pearson_pares.csv': tabela_pearson_pares,
    'previsoes_modelo_final_teste.csv': df_previsoes_teste,
    'tempos_treinamento.csv': df_tempos_treinamento,
}
for nome_csv, esperado in csv_esperados.items():
    caminho_csv = PASTA_DATAFRAME / nome_csv
    assert caminho_csv.is_file(), f'CSV ausente: {nome_csv}'
    pd.testing.assert_frame_equal(
        pd.read_csv(caminho_csv), esperado.reset_index(drop=True),
        check_dtype=False, check_exact=False, rtol=1e-7, atol=1e-10,
    )

def exportar_excel_resultados(caminho, tabelas):

    for nome_aba, tabela in tabelas.items():
        if len(tabela) + 1 > 1_048_576:
            raise ValueError(
                f'{nome_aba} excede o limite de linhas do Excel.'
            )

    with pd.ExcelWriter(
        caminho,
        engine='openpyxl'
    ) as writer:

        for nome_aba, tabela in tabelas.items():

            tabela.to_excel(
                writer,
                sheet_name=nome_aba,
                index=False,
                freeze_panes=(1, 0),
            )

            aba = writer.sheets[nome_aba]
            aba.auto_filter.ref = aba.dimensions

            for coluna in aba.iter_cols(
                min_row=1,
                max_row=1
            ):
                celula = coluna[0]

                aba.column_dimensions[
                    celula.column_letter
                ].width = max(
                    18,
                    len(str(celula.value)) + 3,
                )

            aba.column_dimensions['A'].width = 30

        if 'Cenarios' in writer.sheets:
            writer.sheets[
                'Cenarios'
            ].column_dimensions['D'].width = 95

    # Conferência do arquivo exportado
    with pd.ExcelFile(
        caminho,
        engine='openpyxl'
    ) as arquivo_excel:

        assert arquivo_excel.sheet_names == list(tabelas)

        for nome_aba, esperado in tabelas.items():

            lido = pd.read_excel(
                arquivo_excel,
                sheet_name=nome_aba,
            )

            pd.testing.assert_frame_equal(
                lido,
                esperado.reset_index(drop=True),
                check_dtype=False,
                check_exact=False,
                rtol=1e-7,
                atol=1e-10,
            )

if EXPORTAR_EXCEL:
    caminho_excel = PASTA_DATAFRAME / 'resultados.xlsx'
    exportar_excel_resultados(caminho_excel, {
        'Cenarios': df_mestre,
        'Pearson_Pares': tabela_pearson_pares,
        'Previsoes_Teste': df_previsoes_teste,
    })
    print(f'Excel de consulta: {caminho_excel}')

configuracao_execucao['exportacoes'] = {
    'csv': list(csv_esperados),
    'excel': 'resultados.xlsx' if EXPORTAR_EXCEL else None,
}
# Marca a conclusão somente após verificar as exportações.
configuracao_execucao['status'] = 'concluida'
configuracao_execucao['fim_exportacoes_utc'] = datetime.now(timezone.utc).isoformat()
(PASTA_DATAFRAME / 'configuracao_execucao.json').write_text(
    json.dumps(configuracao_execucao, ensure_ascii=False, indent=2, allow_nan=False),
    encoding='utf-8',
)


## 10. Interpretação e alcance

As métricas quantificam a aproximação da ETo calculada pela fórmula FAO-56 Penman–Monteith, usando uma única seed (42) e a mesma divisão aleatória nos 63 cenários. O cenário final é escolhido exclusivamente pelo RMSE de validação e reutiliza os pesos já treinados. As métricas de teste são apresentadas para avaliação, sem reorientar a seleção.

A normalização global das entradas e do alvo é uma decisão metodológica do estudo. Suas estatísticas incluem treino, validação e teste; portanto, o pré-processamento é compartilhado entre as partições. Os pesos da rede são ajustados com os dados de treino, e o early stopping utiliza a validação.

A tabela auxiliar permite explorar associações entre correlações individuais, redundância entre entradas e erro das MLPs. Pearson mede associação linear; seu valor isolado não determina o desempenho de uma rede não linear nem demonstra causalidade. Todos os cenários são definidos antes do treinamento.

As conclusões se referem à aproximação do alvo calculado na base da estação A601. As combinações avaliam a disponibilidade de entradas para a MLP, considerando que Rn já incorpora informação de outras variáveis. O estudo não constitui validação contra evapotranspiração medida em campo nem demonstra transferência para outras estações.

A implementação e a proveniência dos dados estão documentadas em [FAO-56/README.md](FAO-56/README.md).
